## tl;dr

개방ID `5409221172`는 2009~2025년 중 15개 연도에 경기 수원시 대학원으로 나타난다. 2025년 9개 패널·76행의 실질 측정값은 모두 0이고, 도자공예전공·미술경영전공은 모두 `폐과`다. KEDI에서 `경기대학교 조형대학원`이 2009·2011·2018·2022년에 반복 후보로 나타난다.

## Context & Methods

EDSS DuckDB의 2025년 패널 반환값, 0101 연도 이력, 1017 학과 상태, KEDI 반복 후보를 확인하고 현재 경기대학교 예술대학원 ID와 비교한다.

### Key Assumptions

- 개방ID는 문자열로 처리한다.
- 명시적 0은 결측치와 구분한다.
- KEDI 후보는 직접 학교명 매핑이 아니므로 자동 조인하지 않는다.

In [1]:
import csv
import decimal
import os
from pathlib import Path
import duckdb

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
database_path = Path(os.environ.get(
    'EDSS_DUCKDB_PATH',
    '/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb',
))
identity_path = repo_root / 'data/processed/edss_0101_kedi_openid_identity_2009_2025.csv'
evidence_path = repo_root / 'data/processed/edss_0101_kedi_row_match_evidence_2009_2025.csv'
assert database_path.exists() and identity_path.exists() and evidence_path.exists()
connection = duckdb.connect(str(database_path), read_only=True)
open_id = '5409221172'
current_art_id = '6763784468'
duckdb.__version__, database_path.name

('1.4.1', 'edss_all.duckdb')

## Results

### 1. 2025년 패널 반환값

In [2]:
tables = connection.execute("""
SELECT table_schema, table_name
FROM information_schema.columns
WHERE column_name='개방ID'
  AND table_schema IN ('higher_education', 'university_disclosure')
ORDER BY table_schema, table_name
""").fetchall()
panel_rows = []
for schema_name, table_name in tables:
    row_count = connection.execute(
        f"SELECT COUNT(*) FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    ).fetchone()[0]
    if row_count:
        panel_rows.append((schema_name, table_name, row_count))
assert len(panel_rows) == 9
assert sum(row[2] for row in panel_rows) == 76
panel_rows

[('higher_education', 'panel_0101', 1),
 ('higher_education', 'panel_0104', 1),
 ('higher_education', 'panel_0105', 2),
 ('higher_education', 'panel_0231', 4),
 ('higher_education', 'panel_0246', 12),
 ('university_disclosure', 'panel_0306', 3),
 ('university_disclosure', 'panel_0308', 5),
 ('university_disclosure', 'panel_0715', 36),
 ('university_disclosure', 'panel_1017', 12)]

### 2. 2025년 비영 측정값

In [3]:
dimension_columns = {
    '조사년도', '개방ID', '적용년도', '학기구분명', '수업연한명', '개설기간명',
    '학년명', '성별명', '학위과정구분명', '학과명', '학과한글명', '학과상태명',
    '단과대학명', '교육부계열명', '학과계열구분명', '주야간계절구분명',
    '본분교명', '시도명', '지역명', '학교구분명', '학제유형명', '교원구분명',
}
nonzero_measure_cells = []
for schema_name, table_name, _ in panel_rows:
    cursor = connection.execute(
        f"SELECT * FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    )
    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]
    for column_index, column_name in enumerate(columns):
        if column_name.startswith('_') or column_name in dimension_columns:
            continue
        for row in rows:
            value = row[column_index]
            try:
                numeric_value = decimal.Decimal(str(value).replace(',', ''))
            except decimal.InvalidOperation:
                continue
            if numeric_value != 0:
                nonzero_measure_cells.append((schema_name, table_name, column_name, value))
assert nonzero_measure_cells == []
nonzero_measure_cells

[]

### 3. 학교개황 연도 이력

In [4]:
yearly_rows = connection.execute("""
SELECT 조사년도, 지역명, 본분교명, 고등교육학교_입학생수,
       고등교육학교_졸업생수, 고등교육학교_교원수,
       고등교육학교_재적학생수, 고등교육학교_학과수
FROM higher_education.panel_0101
WHERE 개방ID=?
ORDER BY 조사년도
""", [open_id]).fetchall()
assert len(yearly_rows) == 15
assert {row[0] for row in yearly_rows} == {str(year) for year in range(2009, 2026)} - {'2016', '2017'}
assert {row[1] for row in yearly_rows} == {'경기 수원시'}
nonzero_years = [row for row in yearly_rows if any(value != '0' for value in row[3:])]
nonzero_years

[('2009', '경기 수원시', '본교', '0', '8', '0', '0', '0'),
 ('2011', '경기 수원시', '본교', '0', '1', '0', '0', '0'),
 ('2018', '경기 수원시', '본교', '0', '1', '0', '0', '0'),
 ('2019', '경기 수원시', '본교', '0', '1', '0', '0', '0'),
 ('2022', '경기 수원시', '본교', '0', '1', '0', '0', '0')]

### 4. 폐과 학과

In [5]:
department_rows = connection.execute("""
SELECT DISTINCT 조사년도, 학과명, 학과상태명
FROM university_disclosure.panel_1017
WHERE 개방ID=?
ORDER BY 조사년도, 학과명
""", [open_id]).fetchall()
assert {status for _, _, status in department_rows} == {'폐과'}
latest_departments = [row for row in department_rows if row[0] == '2025']
assert latest_departments == [
    ('2025', '도자공예전공', '폐과'),
    ('2025', '미술경영전공', '폐과'),
]
latest_departments

[('2025', '도자공예전공', '폐과'), ('2025', '미술경영전공', '폐과')]

### 5. KEDI 반복 후보와 현재 예술대학원 비교

In [6]:
with evidence_path.open(encoding='utf-8-sig', newline='') as handle:
    candidate_rows = [row for row in csv.DictReader(handle) if row['openid'] == open_id]
candidate_preview = [
    {key: row[key] for key in ['year', 'kedi_school_name', 'kedi_school_status', 'kedi_region', 'match_families']}
    for row in candidate_rows
]
assert len(candidate_rows) == 4
assert {row['kedi_school_name'] for row in candidate_rows} == {'경기대학교 조형대학원'}
assert {row['year'] for row in candidate_rows} == {'2009', '2011', '2018', '2022'}
candidate_preview

[{'year': '2009',
  'kedi_school_name': '경기대학교 조형대학원',
  'kedi_school_status': '폐교',
  'kedi_region': '경기 수원시',
  'match_families': 'graduates'},
 {'year': '2011',
  'kedi_school_name': '경기대학교 조형대학원',
  'kedi_school_status': '폐교',
  'kedi_region': '경기 수원시',
  'match_families': 'graduates'},
 {'year': '2018',
  'kedi_school_name': '경기대학교 조형대학원',
  'kedi_school_status': '기존',
  'kedi_region': '경기 수원시',
  'match_families': 'graduates'},
 {'year': '2022',
  'kedi_school_name': '경기대학교 조형대학원',
  'kedi_school_status': '폐교',
  'kedi_region': '경기 수원시',
  'match_families': 'graduates'}]

In [7]:
with identity_path.open(encoding='utf-8-sig', newline='') as handle:
    identity_rows = {row['openid']: row for row in csv.DictReader(handle) if row['openid'] in {open_id, current_art_id}}
assert identity_rows[open_id]['identity_status'] == 'unmatched'
assert identity_rows[current_art_id]['latest_direct_school_name'] == '경기대학교 예술대학원'
comparison_rows = connection.execute("""
SELECT 개방ID, 지역명, 고등교육학교_재적학생수, 고등교육학교_학과수, 고등교육학교_교원수
FROM higher_education.panel_0101
WHERE 조사년도='2025' AND 개방ID IN (?, ?)
ORDER BY 개방ID
""", [open_id, current_art_id]).fetchall()
assert comparison_rows == [
    ('5409221172', '경기 수원시', '0', '0', '0'),
    ('6763784468', '경기 수원시 영통구', '37', '5', '1'),
]
comparison_rows

[('5409221172', '경기 수원시', '0', '0', '0'),
 ('6763784468', '경기 수원시 영통구', '37', '5', '1')]

## Takeaways

- `5409221172`는 2025년 9개 패널·76행의 실질 측정값이 모두 0이다.
- 2025년 도자공예전공·미술경영전공은 모두 폐과다.
- KEDI의 `경기대학교 조형대학원`이 네 해에 반복 후보로 나타나며, 2022년 후보 상태는 폐교다.
- 현재 경기대학교 예술대학원은 별도 ID `6763784468`로 2025년 재적학생 37명·학과 5개·교원 1명이다. 대상 ID는 구 조형대학원의 `폐교·폐과 계열 잔존 ID` 후보로 보고 자동 조인하지 않는다.